# Metabolic Health Explorer: Modelos iniciales

### Importaciones

In [32]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import make_scorer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import SplineTransformer, PolynomialFeatures
from sklearn.model_selection import GridSearchCV


RANDOM_STATE = 42

In [4]:
RUTA = Path("..") / "data" / "raw" / "NHANES_consolidado.csv"

df_nhanes = pd.read_csv(RUTA)
adultos = df_nhanes[df_nhanes["Edad"] >= 18].copy()

print("Total de filas", len(df_nhanes))
print("Adultos", len(adultos))
print("Positivos", adultos["Diabetes"].sum())
print("Tasa de positivos", round(adultos["Diabetes"].mean() * 100, 1), "%")

Total de filas 9763
Adultos 6110
Positivos 902
Tasa de positivos 14.8 %


In [5]:
n_ceros = (adultos["Presion_Diastolica"] < 1).sum()
print("Diastólicas en ~0:", n_ceros)

adultos.loc[adultos["Presion_Diastolica"] < 1, "Presion_Diastolica"] = np.nan

# Género: 1 = Hombre, 2 = Mujer en NHANES. Lo pasamos a 0/1 para que sea binaria
adultos["Mujer"] = (adultos["Genero"] == 2).astype(int)

# Porcentaje de nulos por columna
print(adultos.isna().mean().mul(100).round(1).sort_values(ascending=False))

Diastólicas en ~0: 29
Presion_Diastolica    11.7
Presion_Sistolica     11.2
Cintura_cm             8.6
Colesterol_Total       7.7
Colesterol_HDL         7.7
IMC                    4.4
Ingreso_Hogar          1.5
Ingreso_Familia        1.4
Diabetes               0.0
Edad                   0.0
Genero                 0.0
Raza_Etnia             0.0
Mujer                  0.0
dtype: float64


In [6]:
VARS_SIN_LAB = ["Edad", "Mujer", "IMC", "Cintura_cm",
                "Presion_Sistolica", "Presion_Diastolica"]

VARS_CON_LAB = VARS_SIN_LAB + ["Colesterol_Total", "Colesterol_HDL"]

OBJETIVO = "Diabetes"

X = adultos[VARS_CON_LAB] 
y = adultos[OBJETIVO]

print(X.shape, y.shape)

(6110, 8) (6110,)


### Separar entrenamiento y prueba

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,          
    stratify=y,            
    random_state=RANDOM_STATE
)

print("Entrenamiento:", X_train.shape, ", positivos:", y_train.sum(),
      ", tasa:", round(y_train.mean() * 100, 1), "%")
print("Prueba:", X_test.shape,  ", positivos:", y_test.sum(),
      ", tasa:", round(y_test.mean() * 100, 1), "%")

Entrenamiento: (4888, 8) , positivos: 722 , tasa: 14.8 %
Prueba: (1222, 8) , positivos: 180 , tasa: 14.7 %


### Métrica de sensibilidad para el tamizaje al 40% de la población

In [11]:
def recall_at_k(y_true, y_score, k=0.40):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    n_tamizar = int(np.ceil(k * len(y_true)))           
    orden = np.argsort(-y_score, kind="stable")         
    top = orden[:n_tamizar]                            

    return y_true[top].sum() / y_true.sum()             

In [12]:
# Prueba 1: la edad como puntaje
print("Solo edad:", round(recall_at_k(y, X["Edad"]) * 100, 1), "%")

# Prueba 2: puntaje aleatorio
rng = np.random.default_rng(RANDOM_STATE)
print("Aleatorio:", round(recall_at_k(y, rng.random(len(y))) * 100, 1), "%")

# Prueba 3: puntaje perfecto
print("Perfecto:", round(recall_at_k(y, y) * 100, 1), "%")

Solo edad: 72.3 %
Aleatorio: 38.7 %
Perfecto: 100.0 %


#### Convertir la métrica en scorer

In [14]:
scorer_r40 = make_scorer(recall_at_k, response_method="predict_proba")

### Modelos de referencia con cross-validation

In [16]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

METRICAS = {
    "recall_40": scorer_r40,
    "roc_auc": "roc_auc", 
    "pr_auc": "average_precision",
}

In [17]:
resultados = [] 

def evaluar(nombre, modelo, columnas):
    cv_res = cross_validate(modelo, X_train[columnas], y_train,
                            cv=cv, scoring=METRICAS)
    fila = {"modelo": nombre, "n_vars": len(columnas)}
    for m in METRICAS:
        fila[f"{m}_media"] = cv_res[f"test_{m}"].mean()
        fila[f"{m}_std"] = cv_res[f"test_{m}"].std()
    resultados.append(fila)
    return pd.DataFrame([fila]).round(3)

#### Modelo Dummy

In [19]:
dummy = DummyClassifier(strategy="prior")
evaluar("Dummy", dummy, VARS_SIN_LAB)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Dummy,6,0.378,0.032,0.5,0.0,0.148,0.0


#### Solo edad

In [21]:
solo_edad = LogisticRegression()
evaluar("Solo edad", solo_edad, ["Edad"])

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Solo edad,1,0.715,0.029,0.735,0.017,0.262,0.017


In [22]:
pd.DataFrame(resultados).round(3)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Dummy,6,0.378,0.032,0.500,0.000,0.148,0.000
1,Solo edad,1,0.715,0.029,0.735,0.017,0.262,0.017


## Regresión Logística

In [24]:
def pipeline_logistica(class_weight="balanced", C=1.0):
    return Pipeline([
        ("imputar", SimpleImputer(strategy="median")),   
        ("escalar", StandardScaler()), 
        ("modelo", LogisticRegression(class_weight=class_weight,
                                      C=C, max_iter=1000)),
    ])

In [25]:
evaluar("Logística sin lab", pipeline_logistica(), VARS_SIN_LAB)
evaluar("Logística con lab", pipeline_logistica(), VARS_CON_LAB)
pd.DataFrame(resultados).round(3)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Dummy,6,0.378,0.032,0.500,0.000,0.148,0.000
1,Solo edad,1,0.715,0.029,0.735,0.017,0.262,0.017
2,Logística sin lab,6,0.770,0.042,0.783,0.020,0.355,0.036
3,Logística con lab,8,0.795,0.028,0.798,0.017,0.398,0.034


#### Experimento IMC y cintura

In [26]:
sin_cintura = [v for v in VARS_SIN_LAB if v != "Cintura_cm"]
sin_imc     = [v for v in VARS_SIN_LAB if v != "IMC"]

evaluar("Logística sin lab - solo IMC",    pipeline_logistica(), sin_cintura)
evaluar("Logística sin lab - solo cintura", pipeline_logistica(), sin_imc)
pd.DataFrame(resultados).round(3)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Dummy,6,0.378,0.032,0.500,0.000,0.148,0.000
1,Solo edad,1,0.715,0.029,0.735,0.017,0.262,0.017
2,Logística sin lab,6,0.770,0.042,0.783,0.020,0.355,0.036
3,Logística con lab,8,0.795,0.028,0.798,0.017,0.398,0.034
4,Logística sin lab - solo IMC,5,0.767,0.035,0.776,0.018,0.336,0.035
5,Logística sin lab - solo cintura,5,0.767,0.042,0.783,0.021,0.354,0.034


#### Experimento class_weight

In [27]:
evaluar("Logística sin lab - sin class_weight",
        pipeline_logistica(class_weight=None), VARS_SIN_LAB)
pd.DataFrame(resultados).round(3)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Dummy,6,0.378,0.032,0.500,0.000,0.148,0.000
1,Solo edad,1,0.715,0.029,0.735,0.017,0.262,0.017
2,Logística sin lab,6,0.770,0.042,0.783,0.020,0.355,0.036
3,Logística con lab,8,0.795,0.028,0.798,0.017,0.398,0.034
4,Logística sin lab - solo IMC,5,0.767,0.035,0.776,0.018,0.336,0.035
5,Logística sin lab - solo cintura,5,0.767,0.042,0.783,0.021,0.354,0.034
6,Logística sin lab - sin class_weight,6,0.765,0.046,0.784,0.021,0.360,0.034


In [28]:
modelo_sin = pipeline_logistica().fit(X_train[VARS_SIN_LAB], y_train)
modelo_con = pipeline_logistica().fit(X_train[VARS_CON_LAB], y_train)

coefs = pd.DataFrame({
    "sin_lab": pd.Series(modelo_sin.named_steps["modelo"].coef_[0], index=VARS_SIN_LAB),
    "con_lab": pd.Series(modelo_con.named_steps["modelo"].coef_[0], index=VARS_CON_LAB),
}).round(3)
coefs.sort_values("con_lab", key=abs, ascending=False)

,sin_lab,con_lab
Edad,0.913,0.951
Cintura_cm,0.595,0.511
Colesterol_HDL,NaN,-0.379
Colesterol_Total,NaN,-0.221
Mujer,0.062,0.179
Presion_Sistolica,0.152,0.173
IMC,0.113,0.084
Presion_Diastolica,-0.066,-0.016


### Regresión Logística con no linealidades

In [30]:
def pipeline_splines(columnas, vars_spline=("Edad", "Cintura_cm"),
                     interaccion=False, C=1.0, n_knots=5):
    vars_spline = list(vars_spline)
    vars_lineales = [c for c in columnas if c not in vars_spline]

    transformaciones = [
        ("splines", Pipeline([
            ("imputar", SimpleImputer(strategy="median")),
            ("spline", SplineTransformer(n_knots=n_knots, degree=3)),
        ]), vars_spline),

        ("lineales", Pipeline([
            ("imputar", SimpleImputer(strategy="median")),
            ("escalar", StandardScaler()),
        ]), vars_lineales),
    ]

    if interaccion:
        transformaciones.append(
            ("interaccion", Pipeline([
                ("imputar", SimpleImputer(strategy="median")),
                ("escalar", StandardScaler()),
                ("producto", PolynomialFeatures(degree=2, interaction_only=True,
                                                include_bias=False)),
            ]), ["Edad", "Cintura_cm"])
        )

    return Pipeline([
        ("preparar", ColumnTransformer(transformaciones)),
        ("modelo", LogisticRegression(class_weight="balanced", C=C, max_iter=2000)),
    ])

In [31]:
evaluar("Splines sin lab", pipeline_splines(VARS_SIN_LAB), VARS_SIN_LAB)
evaluar("Splines + interacción sin lab",
        pipeline_splines(VARS_SIN_LAB, interaccion=True), VARS_SIN_LAB)
evaluar("Splines + interacción con lab",
        pipeline_splines(VARS_CON_LAB, interaccion=True), VARS_CON_LAB)

pd.DataFrame(resultados).round(3)

,modelo,n_vars,recall_40_media,recall_40_std,roc_auc_media,roc_auc_std,pr_auc_media,pr_auc_std
0,Dummy,6,0.378,0.032,0.500,0.000,0.148,0.000
1,Solo edad,1,0.715,0.029,0.735,0.017,0.262,0.017
2,Logística sin lab,6,0.770,0.042,0.783,0.020,0.355,0.036
3,Logística con lab,8,0.795,0.028,0.798,0.017,0.398,0.034
4,Logística sin lab - solo IMC,5,0.767,0.035,0.776,0.018,0.336,0.035
5,Logística sin lab - solo cintura,5,0.767,0.042,0.783,0.021,0.354,0.034
6,Logística sin lab - sin class_weight,6,0.765,0.046,0.784,0.021,0.360,0.034
7,Splines sin lab,6,0.769,0.022,0.793,0.015,0.367,0.026
8,Splines + interacción sin lab,6,0.770,0.028,0.792,0.016,0.367,0.028
9,Splines + interacción con lab,8,0.798,0.033,0.808,0.015,0.416,0.031


#### Ajuste de hiperparámetros

In [33]:
grid = GridSearchCV(
    pipeline_splines(VARS_SIN_LAB, interaccion=True),
    param_grid={
        "modelo__C": [0.01, 0.1, 1, 10],
        "preparar__splines__spline__n_knots": [3, 4, 5, 7],
    },
    scoring=scorer_r40,
    cv=cv,
)
grid.fit(X_train[VARS_SIN_LAB], y_train)

print("Mejores parámetros:", grid.best_params_)
print("Mejor recall@40 %:", round(grid.best_score_, 3))

(pd.DataFrame(grid.cv_results_)
   [["params", "mean_test_score", "std_test_score"]]
   .sort_values("mean_test_score", ascending=False)
   .head(6))

Mejores parámetros: {'modelo__C': 0.01, 'preparar__splines__spline__n_knots': 4}
Mejor recall@40 %: 0.774


,params,mean_test_score,std_test_score
1,"{'modelo__C': 0.01, 'preparar__splines__spline...",0.774301,0.032194
9,"{'modelo__C': 1, 'preparar__splines__spline__n...",0.772902,0.027409
0,"{'modelo__C': 0.01, 'preparar__splines__spline...",0.771561,0.041735
2,"{'modelo__C': 0.01, 'preparar__splines__spline...",0.771523,0.029570
12,"{'modelo__C': 10, 'preparar__splines__spline__...",0.771513,0.026947
8,"{'modelo__C': 1, 'preparar__splines__spline__n...",0.771504,0.024432
